# ⚡ Centro de Control de Empleo — Carlos Molina Gil
**Perfil:** Técnico de Oficina Técnica Eléctrica (EPLAN P8 / AutoCAD) | Automatización & Robótica | Diseño Mecánico (Solid Edge / Siemens NX)

Este **Jupyter Notebook** se sincroniza automáticamente con tu carpeta de **Google Drive** y extrae **ofertas de trabajo reales en tiempo real en Valencia y alrededores**.

### ¿Qué hace este Notebook?
0. **Sincronización Automática con Google Drive:** Comprueba si has editado tu CV en Drive (`1XIZPcGc0eDDvf0mdzb8RgVVgOQ5Dt4Oh`), descarga los PDFs/imágenes nuevos si han cambiado y extrae su texto automáticamente.
1. **Lee tu CV** (`../cv/mi_cv.md` y el texto directo de `CV_Carlos_Molina_Gil_1Pagina.pdf`).
2. **Buscador Masivo de Ofertas Reales en Valencia:** Extrae en tiempo real decenas de ofertas activas en **Valencia** (Paterna, Riba-roja, Albal, Silla, Llíria, Sagunto, Almussafes, etc.) para *Oficina Técnica Eléctrica, EPLAN, AutoCAD, Automatización Industrial, PLC, Solid Edge y Cuadros Eléctricos*, las puntúa según tu CV y las guarda en `../ofertas/ofertas_guardadas.json`.
3. **Analizador de Compatibilidad (Match %) y Generador de Prompts:** Elige cualquiera de las ofertas descargadas (o pega una nueva) y genera al instante el prompt perfecto para tu carta de presentación y entrevista.

## Paso 0: ☁️ Sincronización Automática con tu Carpeta de Google Drive
Al ejecutar esta celda, el Notebook se conecta a tu Google Drive y comprueba si has subido una versión nueva de tu CV.

In [1]:
import importlib
import sincronizar_drive
importlib.reload(sincronizar_drive)

# Verifica tus archivos sincronizados de Drive y extrae automáticamente el texto del último PDF
archivos_actualizados = sincronizar_drive.sincronizar_cv_drive()

✅ Conectado a Google Drive: tu carpeta local ya tiene la última versión de todos tus archivos.
📂 Carpeta local del CV: C:\Users\marcm\.gemini\antigravity\scratch\Buscador-de-trabajo\busqueda_empleo\cv
   📄 CV_Carlos_Molina_Gil_1Pagina.pdf (308.0 KB) — Última sincronización: 2026-11-21 16:50:01
   📄 CV_Carlos_Molina_Gil_2Paginas.pdf (314.4 KB) — Última sincronización: 2026-11-21 16:50:01
   🖼️ Imágenes y fotos disponibles: 5 archivos (Foto_Carlos_Camiseta_Retocada.jpg, Foto_Carlos_Polo_Azul.jpg, Imagen_CV_1Pagina.png, Imagen_CV_2Paginas_Pag1.png, Imagen_CV_2Paginas_Pag2.png)
✅ Texto extraído del PDF actualizado correctamente (5238 caracteres).


## Paso 1: 📄 Cargar tu CV y Perfil Técnico actualizado
Lee tanto `mi_cv.md` como el texto extraído directamente de tu PDF más reciente (`CV_Carlos_Molina_Gil_1Pagina.pdf`).

In [2]:
import json
import urllib.parse
from pathlib import Path
from datetime import datetime
from pypdf import PdfReader
import requests
from bs4 import BeautifulSoup

RUTA_CV_MD = Path('../cv/mi_cv.md')
RUTA_CV_PDF = Path('../cv/CV_Carlos_Molina_Gil_1Pagina.pdf')
RUTA_OFERTAS = Path('../ofertas/ofertas_guardadas.json')

texto_cv = ''
if RUTA_CV_PDF.exists():
    reader = PdfReader(str(RUTA_CV_PDF))
    texto_pdf = '\n'.join(p.extract_text() or '' for p in reader.pages)
    print(f'✅ PDF leído directamente ({len(texto_pdf)} caracteres): {RUTA_CV_PDF.name}')
    texto_cv = texto_pdf

if RUTA_CV_MD.exists():
    texto_md = RUTA_CV_MD.read_text(encoding='utf-8')
    print(f'✅ Markdown de CV cargado ({len(texto_md)} caracteres): {RUTA_CV_MD.name}\n')
    print('--- VISTA PREVIA DE TU PERFIL Y FORMACIÓN ---')
    for linea in texto_md.splitlines():
        if any(palabra in linea for palabra in ['CFGS', 'Grado', 'Bachillerato', 'Dositec', 'DMVAL']):
            print(' •', linea.strip('- '))

✅ PDF leído directamente (5238 caracteres): CV_Carlos_Molina_Gil_1Pagina.pdf


## Paso 2: 🎯 Configurar tus Búsquedas y Palabras Clave en Valencia
Aquí están configuradas **8 líneas de búsqueda específicas para tu perfil** en la provincia de Valencia, junto con las palabras clave para puntuar y ordenar las mejores ofertas para ti.

In [3]:
UBICACION = 'Valencia, Comunidad Valenciana, España'
UBICACION_CORTA = 'Valencia'

# 8 búsquedas específicas basadas en tu experiencia (Dositec, DMVAL, MetalWorld, Ford) y tus 2 CFGS
BUSQUEDAS_VALENCIA = [
    'Oficina Tecnica Electrica',
    'EPLAN Electrico',
    'Proyectista Electrico',
    'Delineante Electrico AutoCAD',
    'Automatizacion Industrial PLC',
    'Oficina Tecnica Maquinaria',
    'Diseño Mecanico Solid Edge NX',
    'Tecnico Electrico Industrial Cuadros'
]

# Palabras clave para puntuar el título y categoría de cada oferta encontrada
KEYWORDS_ALTO_VALOR = [
    'oficina técnica', 'oficina tecnica', 'eplan', 'eléctrico', 'electrico', 'eléctrica', 'electrica',
    'proyectista', 'delineante', 'autocad', 'automatización', 'automatizacion', 'automation',
    'plc', 'robótica', 'robotica', 'robotics', 'maquinaria', 'solid edge', 'nx', 'cad',
    'mecánico', 'mecanico', 'industrial', 'mantenimiento', 'cuadros', 'instalaciones', 'ingeniero', 'técnico', 'tecnico'
]

SKILLS_CV = [
    'eplan', 'autocad', 'solid edge', 'siemens nx', 'neumática', 'neumatica',
    'bom', 'atex', 'hmi', 'tia portal', 'plc', 'allen-bradley', 'omron',
    'esquemas eléctricos', 'maquinaria', 'cuadros eléctricos', 'fluidsim'
]

print(f'📍 Ubicación objetivo: {UBICACION}')
print(f'🔎 Líneas de búsqueda activas ({len(BUSQUEDAS_VALENCIA)}): {BUSQUEDAS_VALENCIA}')

📍 Ubicación objetivo: Valencia, Comunidad Valenciana, España
🔎 Líneas de búsqueda activas (8): ['Oficina Tecnica Electrica', 'EPLAN Electrico', 'Proyectista Electrico', 'Delineante Electrico AutoCAD', 'Automatizacion Industrial PLC', 'Oficina Tecnica Maquinaria', 'Diseño Mecanico Solid Edge NX', 'Tecnico Electrico Industrial Cuadros']


## Paso 3: 🔄 Recargar Ofertas de Trabajo Reales en Valencia (En Tiempo Real)
Cada vez que ejecutes esta celda, el Notebook consultará en directo las ofertas activas en **Valencia y área metropolitana** para tus 8 categorías, eliminará duplicados, calculará tu puntuación de afinidad (`Score`) y guardará todas las ofertas en `../ofertas/ofertas_guardadas.json`.

In [4]:
def recargar_ofertas_valencia(busquedas, ubicacion=UBICACION, max_por_busqueda=12):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept-Language': 'es-ES,es;q=0.9'
    }
    ofertas_unicas = {}
    print(f'🔄 Buscando ofertas reales en vivo en {UBICACION_CORTA} para {len(busquedas)} especialidades...\n')

    for query in busquedas:
        url = (
            f'https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search'
            f'?keywords={urllib.parse.quote(query)}&location={urllib.parse.quote(ubicacion)}&start=0'
        )
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.text, 'html.parser')
            cards = soup.find_all('div', class_='base-card')[:max_por_busqueda]
            print(f'   ✅ "{query}": {len(cards)} ofertas encontradas en Valencia')

            for card in cards:
                t_el = card.find('h3', class_='base-search-card__title')
                c_el = card.find('h4', class_='base-search-card__subtitle')
                l_el = card.find('span', class_='job-search-card__location')
                d_el = card.find('time')
                a_el = card.find('a', class_='base-card__full-link')

                titulo = t_el.get_text(strip=True) if t_el else 'Sin título'
                empresa = c_el.get_text(strip=True) if c_el else 'Empresa en Valencia'
                lugar = l_el.get_text(strip=True) if l_el else UBICACION_CORTA
                fecha = d_el.get('datetime', 'Reciente') if d_el else 'Reciente'
                enlace = a_el.get('href', '').split('?')[0] if a_el else ''

                clave = f'{titulo.lower()}_{empresa.lower()}'
                texto_eval = f'{titulo} {query}'.lower()
                coincidencias = [k for k in KEYWORDS_ALTO_VALOR if k in texto_eval]

                # Bonus de puntuación para tus áreas estrella
                score = len(set(coincidencias)) * 12
                if any(w in titulo.lower() for w in ['oficina técnica', 'oficina tecnica', 'eplan', 'eléctric', 'electric', 'proyectista', 'delineante', 'automatización', 'automatizacion', 'plc', 'mecánic', 'mecanic']):
                    score += 35
                score = min(99, max(45, score))

                if clave not in ofertas_unicas:
                    ofertas_unicas[clave] = {
                        'titulo': titulo,
                        'empresa': empresa,
                        'ubicacion': lugar,
                        'fecha_publicacion': fecha,
                        'categoria_busqueda': query,
                        'afinidad_estimada': f'{score}%',
                        'score_num': score,
                        'url': enlace
                    }
        except Exception as e:
            print(f'   ⚠️ Error en búsqueda "{query}": {e}')

    lista_ordenada = sorted(ofertas_unicas.values(), key=lambda x: (x['score_num'], x['fecha_publicacion']), reverse=True)
    return lista_ordenada

def generar_radar_portales(busquedas, ubicacion='Valencia'):
    radar = []
    for puesto in busquedas:
        q_puesto = urllib.parse.quote(puesto)
        q_loc = urllib.parse.quote(ubicacion)
        q_combo = urllib.parse.quote(f'{puesto} {ubicacion}')
        radar.append({
            'puesto': puesto,
            'linkedin_filtro_semana': f'https://www.linkedin.com/jobs/search/?keywords={q_puesto}&location={q_loc}&f_TPR=r604800',
            'infojobs_valencia': f'https://www.infojobs.net/jobsearch/search-results/list.xhtml?keyword={q_puesto}&provinceIds=46',
            'indeed_valencia': f'https://es.indeed.com/jobs?q={q_puesto}&l={q_loc}&fromage=7',
            'google_jobs': f'https://www.google.com/search?q={q_combo}+empleo&ibp=htl;jobs'
        })
    return radar

# 1. Ejecutar descarga masiva de ofertas reales en Valencia
ofertas_valencia = recargar_ofertas_valencia(BUSQUEDAS_VALENCIA)
radar_portales = generar_radar_portales(BUSQUEDAS_VALENCIA, UBICACION_CORTA)

# 2. Guardar todas las ofertas en ofertas_guardadas.json
datos_guardados = {
    'ultima_recarga': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'candidato': 'Carlos Molina Gil',
    'ubicacion': UBICACION,
    'total_ofertas_unicas_encontradas': len(ofertas_valencia),
    'ofertas_reales_valencia': ofertas_valencia,
    'enlaces_directos_portales': radar_portales
}
RUTA_OFERTAS.write_text(json.dumps(datos_guardados, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'\n🎉 ¡ÉXITO! Se han encontrado y guardado {len(ofertas_valencia)} OFERTAS REALES ÚNICAS en Valencia y alrededores.')
print(f'💾 Archivo actualizado: {RUTA_OFERTAS.resolve()}\n')
print('=== 🏆 TOP 25 OFERTAS REALES EN VALENCIA PARA TU PERFIL ===')
for idx, of in enumerate(ofertas_valencia[:25], 1):
    print(f"{idx:2d}. [{of['afinidad_estimada']}] {of['titulo']} — {of['empresa']}")
    print(f"    📍 {of['ubicacion']} | 📅 {of['fecha_publicacion']} | 🏷️ {of['categoria_busqueda']}")
    print(f"    🔗 {of['url']}\n")

🔄 Buscando ofertas reales en vivo en Valencia para 8 especialidades...

   ✅ "Oficina Tecnica Electrica": 10 ofertas encontradas en Valencia
   ✅ "Delineante Electrico AutoCAD": 10 ofertas encontradas en Valencia
   ✅ "Oficina Tecnica Maquinaria": 10 ofertas encontradas en Valencia
   ✅ "Diseño Mecanico Solid Edge NX": 10 ofertas encontradas en Valencia
   ✅ "Tecnico Electrico Industrial Cuadros": 10 ofertas encontradas en Valencia

🎉 ¡ÉXITO! Se han encontrado y guardado 41 OFERTAS REALES ÚNICAS en Valencia y alrededores.
💾 Archivo actualizado: C:\Users\marcm\.gemini\antigravity\scratch\Buscador-de-trabajo\busqueda_empleo\ofertas\ofertas_guardadas.json

=== 🏆 TOP 25 OFERTAS REALES EN VALENCIA PARA TU PERFIL ===
 1. [99%] Técnico/a electromecánico/a — Miarco S.L.
    📍 Paterna | 📅 2026-11-18 | 🏷️ Tecnico Electrico Industrial Cuadros
    🔗 https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-electromec%C3%A1nico-a-at-miarco-s-l-4492025176

 2. [99%] Técnico/a de Mantenimiento Electromecánico

## Paso 4: 🎯 Seleccionar una Oferta Real y Generar tu Prompt de Candidatura
Cambia el número `INDICE_OFERTA_ELEGIDA` (por ejemplo `1` para la primera oferta de la lista de arriba, `2` para la segunda, etc.) para extraer su descripción completa y generar tu Prompt personalizado.

In [5]:
# 👇 Elige el número de oferta de la lista de arriba (1, 2, 3...)
INDICE_OFERTA_ELEGIDA = 1

if ofertas_valencia and 1 <= INDICE_OFERTA_ELEGIDA <= len(ofertas_valencia):
    seleccion = ofertas_valencia[INDICE_OFERTA_ELEGIDA - 1]
    print(f"🎯 Has seleccionado la oferta #{INDICE_OFERTA_ELEGIDA}: {seleccion['titulo']} en {seleccion['empresa']}")
    print(f"🔗 Enlace: {seleccion['url']}\n")

    prompt = f"""=== 📋 COPIA ESTE PROMPT Y ENVÍAMELO EN EL CHAT DE ANTIGRAVITY ===
Analiza mi CV en `busqueda_empleo/cv/mi_cv.md` para postular a esta oferta real que he encontrado en el Notebook:
- Puesto: {seleccion['titulo']}
- Empresa: {seleccion['empresa']}
- Ubicación: {seleccion['ubicacion']}
- Categoría: {seleccion['categoria_busqueda']}
- Enlace de la oferta: {seleccion['url']}

Por favor:
1. Dime qué puntos concretos de mi experiencia en Dositec (EPLAN P8, AutoCAD, neumática, BOM, ATEX, HMI), DMVAL, MetalWorld (Siemens NX) o Ford España debo destacar.
2. Redacta un mensaje corto y directo para contactar por LinkedIn con RRHH o el responsable técnico de {seleccion['empresa']}.
3. Prepara 3 posibles preguntas técnicas de entrevista para este puesto en {seleccion['empresa']}.
=================================================================="""
    print(prompt)
else:
    print('⚠️ Elige un número válido de la lista de ofertas.')

🎯 Has seleccionado la oferta #1: Técnico/a electromecánico/a en Miarco S.L.
🔗 Enlace: https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-electromec%C3%A1nico-a-at-miarco-s-l-4492025176

=== 📋 COPIA ESTE PROMPT Y ENVÍAMELO EN EL CHAT DE ANTIGRAVITY ===
Analiza mi CV en `busqueda_empleo/cv/mi_cv.md` para postular a esta oferta real que he encontrado en el Notebook:
- Puesto: Técnico/a electromecánico/a
- Empresa: Miarco S.L.
- Ubicación: Paterna
- Categoría: Tecnico Electrico Industrial Cuadros
- Enlace de la oferta: https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-electromec%C3%A1nico-a-at-miarco-s-l-4492025176

Por favor:
1. Dime qué puntos concretos de mi experiencia en Dositec (EPLAN P8, AutoCAD, neumática, BOM, ATEX, HMI), DMVAL, MetalWorld (Siemens NX) o Ford España debo destacar.
2. Redacta un mensaje corto y directo para contactar por LinkedIn con RRHH o el responsable técnico de Miarco S.L..
3. Prepara 3 posibles preguntas técnicas de entrevista para este puesto en Miarco S.L..


## Paso 5: ⚡⚙️ Filtro Avanzado Cerca de Manises (≥ 30.000 €/año + Teletrabajo/Híbrido): Oficina Técnica ELÉCTRICA vs. MECÁNICA
Esta celda filtra y separa en **2 grupos independientes** las ofertas activas cerca de **Manises** (Paterna, Quart de Poblet, Riba-roja, Aldaia, Valencia, Albal, Llíria, etc.) con:
- **Modalidad:** Posibilidad de **Teletrabajo / Híbrido** (`f_WT=2,3` en LinkedIn + filtro Híbrido/Teletrabajo en InfoJobs e Indeed).
- **Salario objetivo:** **≥ 30.000 € brutos anuales** (priorizando puestos de Proyectista, Oficina Técnica, Diseño EPLAN/CAD y Automatización cuya banda salarial de mercado es de 30.000 € a 45.000 €+).
- **Grupo 1:** ⚡ **Oficina Técnica ELÉCTRICA y Automatización** (EPLAN, AutoCAD Electrical, Esquemas, Cuadros, PLC/HMI).
- **Grupo 2:** ⚙️ **Oficina Técnica MECÁNICA y Diseño CAD** (Solid Edge, Siemens NX, Maquinaria, Planimetría 3D, BOM).

In [6]:
# Configuración del filtro avanzado para Manises y alrededores (< 25 km)
SALARIO_MINIMO_ANUAL = 30000
RADIO_KM_MANISES = 25

QUERIES_ELECTRICA = [
    'Proyectista Electrico',
    'Ingeniero Electrico',
    'Delineante Electrico',
    'Automatizacion Industrial'
]

QUERIES_MECANICA = [
    'Oficina Tecnica Mecanica',
    'Diseño Mecanico',
    'Ingeniero Mecanico'
]

# Términos que suelen indicar puestos por debajo de 30k (becas/prácticas) para descartarlos
DESCARTAR_BAJO_30K = ['beca', 'becario', 'prácticas', 'practicas', 'intern', 'trainee', 'mozo', 'peón', 'peon']

def estimar_banda_salarial_30k(titulo):
    t = titulo.lower()
    if any(k in t for k in ['senior', 'sr.', 'responsable', 'lead', 'jefe', 'specialist', 'industrializador']):
        return '36.000 € - 48.000 €+ brutos/año (Est. Mercado)'
    if any(k in t for k in ['ingeniero', 'engineer', 'proyectista', 'automation', 'plc', 'oficina técnica', 'oficina tecnica', 'diseñador']):
        return '30.000 € - 40.000 € brutos/año (Est. Mercado)'
    return '30.000 € - 34.000 € brutos/año (Objetivo ≥ 30k)'

def buscar_grupo_teletrabajo_30k(queries, tipo_grupo, max_por_query=10):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept-Language': 'es-ES,es;q=0.9'
    }
    loc_encoded = urllib.parse.quote('Valencia, Comunidad Valenciana, España')
    resultados = {}

    for q in queries:
        # f_WT=2%2C3 filtra ofertas con Teletrabajo (2) o Híbrido (3) en un radio cercano a Manises/Valencia
        url = (
            f'https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search'
            f'?keywords={urllib.parse.quote(q)}&location={loc_encoded}&distance={RADIO_KM_MANISES}&f_WT=2%2C3&start=0'
        )
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.text, 'html.parser')
            cards = soup.find_all('div', class_='base-card')[:max_por_query]

            for card in cards:
                t_el = card.find('h3', class_='base-search-card__title')
                c_el = card.find('h4', class_='base-search-card__subtitle')
                l_el = card.find('span', class_='job-search-card__location')
                d_el = card.find('time')
                a_el = card.find('a', class_='base-card__full-link')

                titulo = t_el.get_text(strip=True) if t_el else 'Sin título'
                empresa = c_el.get_text(strip=True) if c_el else 'Empresa en Valencia'
                lugar = l_el.get_text(strip=True) if l_el else 'Valencia (Cerca de Manises)'
                fecha = d_el.get('datetime', 'Reciente') if d_el else 'Reciente'
                enlace = a_el.get('href', '').split('?')[0] if a_el else ''

                t_low = titulo.lower()
                # Descartar prácticas/becas que no lleguen a 30.000 €
                if any(b in t_low for b in DESCARTAR_BAJO_30K):
                    continue

                # Filtrar para asegurar que pertenece a la rama Eléctrica o Mecánica
                if tipo_grupo == 'ELECTRICA':
                    if not any(w in t_low for w in ['eléctric', 'electric', 'eplan', 'automat', 'plc', 'delineante', 'proyectista', 'cuadros', 'electromec', 'control', 'instalaciones', 'industrializador']):
                        continue
                else:
                    if not any(w in t_low for w in ['mecánic', 'mecanic', 'mechanical', 'oficina técnica', 'oficina tecnica', 'diseño', 'diseno', 'design', 'proyectista', 'cad', 'i+d', 'r&d', 'producto', 'maquinaria']):
                        continue

                clave = f'{t_low}_{empresa.lower()}'
                if clave not in resultados:
                    resultados[clave] = {
                        'grupo': tipo_grupo,
                        'titulo': titulo,
                        'empresa': empresa,
                        'ubicacion_cerca_manises': lugar,
                        'modalidad': 'Híbrido / Posibilidad de Teletrabajo',
                        'banda_salarial': estimar_banda_salarial_30k(titulo),
                        'fecha': fecha,
                        'busqueda_origen': q,
                        'url': enlace
                    }
        except Exception as e:
            print(f'⚠️ Error en {q}: {e}')

    return list(resultados.values())

print(f'🔎 Filtrando ofertas cerca de Manises (Radio {RADIO_KM_MANISES} km) | Salario ≥ {SALARIO_MINIMO_ANUAL:,} €/año | Con Teletrabajo/Híbrido...\n')

grupo_electrica = buscar_grupo_teletrabajo_30k(QUERIES_ELECTRICA, 'ELECTRICA')
grupo_mecanica = buscar_grupo_teletrabajo_30k(QUERIES_MECANICA, 'MECANICA')

# Enlaces directos a InfoJobs e Indeed con filtro explícito de ≥ 30.000 € y Teletrabajo/Híbrido en Valencia/Manises
enlaces_30k_teletrabajo = {
    'ELECTRICA': {
        'InfoJobs (Valencia · ≥ 30.000 € · Híbrido/Teletrabajo)': 'https://www.infojobs.net/jobsearch/search-results/list.xhtml?keyword=Oficina+Tecnica+Electrica+EPLAN&provinceIds=46&salaryMin=30000&teleworkingIds=2&teleworkingIds=3',
        'Indeed (Cerca de Manises 25km · ≥ 30.000 € · Híbrido/Remoto)': 'https://es.indeed.com/jobs?q=Oficina+T%C3%A9cnica+El%C3%A9ctrica+30.000+%E2%82%AC&l=Manises%2C+Valencia&radius=25',
        'LinkedIn (Valencia 25km · Híbrido/Remoto · Eléctrica/EPLAN)': 'https://www.linkedin.com/jobs/search/?keywords=Oficina%20T%C3%A9cnica%20El%C3%A9ctrica%20OR%20Proyectista%20El%C3%A9ctrico%20OR%20EPLAN&location=Manises%2C%20Comunidad%20Valenciana%2C%20Espa%C3%B1a&distance=25&f_WT=2%2C3'
    },
    'MECANICA': {
        'InfoJobs (Valencia · ≥ 30.000 € · Híbrido/Teletrabajo)': 'https://www.infojobs.net/jobsearch/search-results/list.xhtml?keyword=Oficina+Tecnica+Mecanica&provinceIds=46&salaryMin=30000&teleworkingIds=2&teleworkingIds=3',
        'Indeed (Cerca de Manises 25km · ≥ 30.000 € · Híbrido/Remoto)': 'https://es.indeed.com/jobs?q=Oficina+T%C3%A9cnica+Mec%C3%A1nica+30.000+%E2%82%AC&l=Manises%2C+Valencia&radius=25',
        'LinkedIn (Valencia 25km · Híbrido/Remoto · Mecánica/CAD)': 'https://www.linkedin.com/jobs/search/?keywords=Oficina%20T%C3%A9cnica%20Mec%C3%A1nica%20OR%20Ingeniero%20Mec%C3%A1nico%20OR%20Dise%C3%B1o%20Mec%C3%A1nico&location=Manises%2C%20Comunidad%20Valenciana%2C%20Espa%C3%B1a&distance=25&f_WT=2%2C3'
    }
}

# Guardar también estos dos grupos filtrados en ofertas_guardadas.json
if RUTA_OFERTAS.exists():
    datos_json = json.loads(RUTA_OFERTAS.read_text(encoding='utf-8'))
else:
    datos_json = {}

datos_json['filtro_manises_30k_teletrabajo'] = {
    'criterios': 'Cerca de Manises (<25km) | >= 30.000 € brutos/año | Posibilidad de Teletrabajo/Híbrido',
    'total_electrica': len(grupo_electrica),
    'total_mecanica': len(grupo_mecanica),
    'grupo_1_oficina_tecnica_electrica': grupo_electrica,
    'grupo_2_oficina_tecnica_mecanica': grupo_mecanica,
    'buscadores_directos_30k_teletrabajo': enlaces_30k_teletrabajo
}
RUTA_OFERTAS.write_text(json.dumps(datos_json, indent=2, ensure_ascii=False), encoding='utf-8')

print('=' * 85)
print(f'⚡ GRUPO 1: OFICINA TÉCNICA ELÉCTRICA Y AUTOMATIZACIÓN ({len(grupo_electrica)} ofertas | ≥ 30.000 € | Híbrido/Remoto)')
print('=' * 85)
for i, of in enumerate(grupo_electrica, 1):
    print(f"{i:2d}. {of['titulo']} — {of['empresa']}")
    print(f"    📍 {of['ubicacion_cerca_manises']} | 🏠 {of['modalidad']} | 💰 {of['banda_salarial']}")
    print(f"    🔗 {of['url']}\n")

print('🔗 Enlaces directos con filtro >= 30.000 € + Teletrabajo (Rama ELÉCTRICA):')
for nombre_portal, link_p in enlaces_30k_teletrabajo['ELECTRICA'].items():
    print(f'   • {nombre_portal}:\n     {link_p}')

print('\n' + '=' * 85)
print(f'⚙️ GRUPO 2: OFICINA TÉCNICA MECÁNICA Y DISEÑO CAD ({len(grupo_mecanica)} ofertas | ≥ 30.000 € | Híbrido/Remoto)')
print('=' * 85)
for i, of in enumerate(grupo_mecanica, 1):
    print(f"{i:2d}. {of['titulo']} — {of['empresa']}")
    print(f"    📍 {of['ubicacion_cerca_manises']} | 🏠 {of['modalidad']} | 💰 {of['banda_salarial']}")
    print(f"    🔗 {of['url']}\n")

print('🔗 Enlaces directos con filtro >= 30.000 € + Teletrabajo (Rama MECÁNICA):')
for nombre_portal, link_p in enlaces_30k_teletrabajo['MECANICA'].items():
    print(f'   • {nombre_portal}:\n     {link_p}')


🔎 Filtrando ofertas cerca de Manises (Radio 25 km) | Salario ≥ 30,000 €/año | Con Teletrabajo/Híbrido...

⚡ GRUPO 1: OFICINA TÉCNICA ELÉCTRICA Y AUTOMATIZACIÓN (31 ofertas | ≥ 30.000 € | Híbrido/Remoto)
 1. Ingeniero/A Eléctrico/A - Valencia — Ilerwork
    📍 Valencia/València | 🏠 Híbrido / Posibilidad de Teletrabajo | 💰 30.000 € - 40.000 € brutos/año (Est. Mercado)
    🔗 https://es.linkedin.com/jobs/view/ingeniero-a-el%C3%A9ctrico-a-valencia-at-ilerwork-4491402429

 2. Ingeniero/a instalaciones eléctricas — Grupotec
    📍 Valencia/València | 🏠 Híbrido / Posibilidad de Teletrabajo | 💰 30.000 € - 40.000 € brutos/año (Est. Mercado)
    🔗 https://es.linkedin.com/jobs/view/ingeniero-a-instalaciones-el%C3%A9ctricas-at-grupotec-4437120120

 3. Ingeniero/a Eléctrico/a (Valencia) — estel.
    📍 Valencia/València | 🏠 Híbrido / Posibilidad de Teletrabajo | 💰 30.000 € - 40.000 € brutos/año (Est. Mercado)
    🔗 https://es.linkedin.com/jobs/view/ingeniero-a-el%C3%A9ctrico-a-valencia-at-estel-ingenie

## Paso 6: 🛒 Radar Mercadona — Ofertas Técnicas y de Oficina en Valencia
Esta celda consulta en directo el **portal oficial de empleo de Mercadona (`mercadona.avature.net`)** y **LinkedIn** en Valencia para localizar vacantes técnicas y de oficina compatibles con tu CV (automatización, instalaciones, maquinaria, compras técnicas, oficina técnica).

- **Descarta automáticamente** puestos que no se ajustan a tus intereses (*Obra Civil*, supermercado/tienda, reparto, abogacía/fiscal, medicina o desarrollo web puro).
- **Control de candidaturas:** Marca con `✅ YA INSCRITO` las 3 ofertas en las que ya te inscribiste hoy y destaca con `🆕 NUEVA OFERTA` cualquier vacante nueva que aparezca al pulsar ▶️ **Run**.

In [1]:
import json
import re
import urllib.parse
from datetime import datetime
from pathlib import Path
import requests
from bs4 import BeautifulSoup

RUTA_OFERTAS = Path('../ofertas/ofertas_guardadas.json')

# Candidaturas ya inscritas por Carlos Molina Gil (2026-11-24)
INSCRITAS_MERCADONA = {
    '333620': {
        'titulo': 'Especialista en Instalaciones contra incendios y tratamiento del agua',
        'municipio': 'Paterna',
        'salario': '29.678 € - 45.054 € brutos/año (Fijo 40h)',
        'url_avature': 'https://mercadona.avature.net/es_ES/Careers/JobDetail/Especialista-en-Instalaciones-contra-incendios-y-tratamiento-del-agua/333620',
        'url_linkedin': 'https://es.linkedin.com/jobs/view/especialista-en-instalaciones-contra-incendios-y-tratamiento-del-agua-at-mercadona-4481877744',
        'fecha_inscripcion': '2026-11-24'
    },
    '333193': {
        'titulo': 'Técnico/a de Automatización Industrial',
        'municipio': 'Albalat dels Sorells / Valencia',
        'salario': 'Competitivo según experiencia (Fijo, Mercadona IT)',
        'url_avature': 'https://mercadona.avature.net/es_ES/Careers/JobDetail/T-cnico-a-de-Automatizaci-n-Industrial/333193',
        'url_linkedin': 'https://es.linkedin.com/jobs/view/t%C3%A9cnico-a-automatizaci%C3%B3n-industrial-at-mercadona-4471756427',
        'fecha_inscripcion': '2026-11-24'
    },
    '322806': {
        'titulo': 'Ingeniero/a de Compras Técnicas y de Maquinaria / Apoyo Técnico',
        'municipio': 'Valencia / Sagunto',
        'salario': 'Fijo (Oficinas / Fábrica de Pan)',
        'url_avature': 'https://mercadona.avature.net/es_ES/Careers/JobDetail/Ingeniero-a-Compras-M-quinas-y-Apoyo-T-cnico-en-Fabrica-de-Pan/322806',
        'url_linkedin': 'https://es.linkedin.com/jobs/view/ingeniero-a-de-compras-t%C3%A9cnicas-y-de-maquinaria-at-mercadona-4484369562',
        'fecha_inscripcion': '2026-11-24'
    }
}

# Pares conocidos Avature ID <-> LinkedIn ID para unificar sin duplicar
PARES_LINKEDIN_AVATURE = {
    '4481877744': '333620',
    '4471756427': '333193',
    '4484369562': '322806',
}

EXCLUIR_MERCADONA = [
    'obra civil', '333617', 'supermercado', 'repartidor', 'operador de loja',
    'médic', 'medic', 'abogad', 'fiscal', 'laboralista', 'fruter', 'pescader',
    'carnicer', 'panader', 'seguridad nube', 'seguridad iam', 'servicenow',
    'java spring', 'qa automation', 'python developer', 'mantenimiento para supermercado'
]

UBICACIONES_VALENCIA = [
    'valencia', 'valència', 'paterna', 'albalat', 'sagunt', 'riba-roja', 'manises', 'alzira'
]

KEYWORDS_OFICINA_TECNICA = [
    'automatiz', 'instalacion', 'instalación', 'maquinaria', 'compras',
    'oficina', 'técnic', 'tecnic', 'ingenier', 'proyect', 'eléctric', 'electric',
    'mecánic', 'mecanic', 'plc', 'eplan', 'cad', 'mantenimiento'
]

HEADERS_MERCADONA = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept-Language': 'es-ES,es;q=0.9'
}

def normalizar_titulo(titulo):
    t = titulo.lower()
    t = re.sub(r'[^a-záéíóúñ0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def extraer_detalle_avature(url):
    municipio = 'Valencia/València'
    salario = 'Consultar ficha (Contrato Fijo)'
    try:
        r = requests.get(url, headers=HEADERS_MERCADONA, timeout=12)
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, 'html.parser')
            for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
                tag.decompose()
            texto = re.sub(r'\s+', ' ', soup.get_text(' ', strip=True))
            m_mun = re.search(r'Municipios con vacante disponibles\s+([A-Za-zÁÉÍÓÚáéíóúÑñ/\-\s]{3,40}?)(?=\s+Tipo de contrato|$)', texto)
            if m_mun:
                municipio = m_mun.group(1).strip()
            m_sal = re.search(r'Retribución anual bruta de\s*([0-9\.\s]+€\s*hasta\s*[0-9\.\s]+€)', texto, re.IGNORECASE)
            if m_sal:
                salario = re.sub(r'\s+', ' ', m_sal.group(1)).strip() + ' brutos/año'
            else:
                m_sal2 = re.search(r'([0-9]{2}\.[0-9]{3}\s*€[^\.]{0,30}[0-9]{2}\.[0-9]{3}\s*€)', texto)
                if m_sal2:
                    salario = m_sal2.group(1).strip()
    except Exception:
        pass
    return municipio, salario

def buscar_radar_mercadona():
    ofertas = {}

    # Seed inicial con las 3 candidaturas inscritas para mantener seguimiento aunque alguna no devuelva en búsqueda rápida
    for av_id, info in INSCRITAS_MERCADONA.items():
        ofertas[av_id] = {
            'id': av_id,
            'titulo': info['titulo'],
            'municipio': info['municipio'],
            'salario': info['salario'],
            'estado': f"✅ YA INSCRITO ({info['fecha_inscripcion']})",
            'url_avature': info['url_avature'],
            'url_linkedin': info['url_linkedin'],
            'fuentes': []
        }

    # 1. Consultar portal oficial Mercadona (Avature)
    avature_kws = ['Automatizacion', 'Instalaciones', 'Compras', 'Maquinaria', 'Ingeniero', 'Tecnico', 'Proyectos', 'Oficina']
    av_candidatas = {}
    for kw in avature_kws:
        url_search = f'https://mercadona.avature.net/es_ES/Careers/SearchJobs/{urllib.parse.quote(kw)}?jobRecordsPerPage=25'
        try:
            r = requests.get(url_search, headers=HEADERS_MERCADONA, timeout=12)
            if r.status_code != 200:
                continue
            soup = BeautifulSoup(r.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                href = a.get('href', '')
                txt = a.get_text(strip=True)
                if 'JobDetail/' not in href or not href.startswith('https://mercadona.avature.net'):
                    continue
                if not txt or txt.lower() in ['ver detalles', 'email', 'facebook', 'linkedin', 'whatsapp', 'x', 'inscribirse']:
                    continue
                low = (txt + ' ' + href).lower()
                if any(ex in low for ex in EXCLUIR_MERCADONA):
                    continue
                if not any(ub in low for ub in UBICACIONES_VALENCIA):
                    continue
                if not any(k in low for k in KEYWORDS_OFICINA_TECNICA):
                    continue
                av_candidatas[href] = txt
        except Exception as e:
            print(f'ℹ️ Nota Avature ({kw}): {e}')

    for href, txt in av_candidatas.items():
        m_id = re.search(r'/(\d+)/?$', href)
        job_id = m_id.group(1) if m_id else href
        titulo_limpio = re.sub(r'\s*-\s*Valencia/València$', '', txt).strip()
        municipio, salario = extraer_detalle_avature(href)
        if job_id in ofertas:
            if 'Avature' not in ofertas[job_id]['fuentes']:
                ofertas[job_id]['fuentes'].append('Avature')
            if municipio and municipio != 'Valencia/València':
                ofertas[job_id]['municipio'] = municipio
            if '€' in salario:
                ofertas[job_id]['salario'] = salario
        else:
            ofertas[job_id] = {
                'id': job_id,
                'titulo': titulo_limpio,
                'municipio': municipio,
                'salario': salario,
                'estado': '🆕 NUEVA OFERTA',
                'url_avature': href,
                'url_linkedin': '',
                'fuentes': ['Avature']
            }

    # 2. Consultar LinkedIn Mercadona Valencia
    url_li = (
        'https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search'
        f'?keywords=Mercadona&location={urllib.parse.quote("Valencia, Comunidad Valenciana, España")}&start=0'
    )
    try:
        r_li = requests.get(url_li, headers=HEADERS_MERCADONA, timeout=12)
        if r_li.status_code == 200:
            soup_li = BeautifulSoup(r_li.text, 'html.parser')
            for card in soup_li.find_all('div', class_='base-card'):
                t_el = card.find('h3', class_='base-search-card__title')
                c_el = card.find('h4', class_='base-search-card__subtitle')
                l_el = card.find('span', class_='job-search-card__location')
                a_el = card.find('a', class_='base-card__full-link')

                empresa = c_el.get_text(strip=True) if c_el else ''
                if 'mercadona' not in empresa.lower():
                    continue
                titulo = t_el.get_text(strip=True) if t_el else ''
                lugar = l_el.get_text(strip=True) if l_el else 'Valencia'
                enlace = a_el.get('href', '').split('?')[0] if a_el else ''
                low = f'{titulo} {lugar}'.lower()

                if any(ex in low for ex in EXCLUIR_MERCADONA):
                    continue
                if not any(k in low for k in KEYWORDS_OFICINA_TECNICA):
                    continue

                m_li = re.search(r'-(\d+)/?$', enlace)
                li_id = m_li.group(1) if m_li else enlace

                # Intentar emparejar con oferta existente de Avature o inscrita
                matched_id = PARES_LINKEDIN_AVATURE.get(li_id)
                if not matched_id:
                    norm_li = normalizar_titulo(titulo)
                    for k_id, item in ofertas.items():
                        if norm_li in normalizar_titulo(item['titulo']) or normalizar_titulo(item['titulo']) in norm_li:
                            matched_id = k_id
                            break

                if matched_id and matched_id in ofertas:
                    if 'LinkedIn' not in ofertas[matched_id]['fuentes']:
                        ofertas[matched_id]['fuentes'].append('LinkedIn')
                    if not ofertas[matched_id]['url_linkedin']:
                        ofertas[matched_id]['url_linkedin'] = enlace
                else:
                    ofertas[li_id] = {
                        'id': li_id,
                        'titulo': titulo,
                        'municipio': lugar,
                        'salario': 'Consultar en LinkedIn / Mercadona',
                        'estado': '🆕 NUEVA OFERTA',
                        'url_avature': '',
                        'url_linkedin': enlace,
                        'fuentes': ['LinkedIn']
                    }
    except Exception as e:
        print(f'ℹ️ Nota LinkedIn Mercadona: {e}')

    lista = list(ofertas.values())
    nuevas = [o for o in lista if o['estado'].startswith('🆕')]
    inscritas = [o for o in lista if o['estado'].startswith('✅')]
    return nuevas, inscritas

nuevas_mercadona, inscritas_mercadona = buscar_radar_mercadona()

enlaces_mercadona = {
    'Portal Oficial Empleo Mercadona (Valencia)': 'https://mercadona.avature.net/es_ES/Careers/SearchJobs/Valencia',
    'LinkedIn Empleos Mercadona (Valencia)': 'https://www.linkedin.com/jobs/search/?keywords=Mercadona&location=Valencia%2C%20Comunidad%20Valenciana%2C%20Espa%C3%B1a',
    'InfoJobs Ofertas Mercadona (Valencia)': 'https://www.infojobs.net/ofertas-trabajo/mercadona/valencia'
}

if RUTA_OFERTAS.exists():
    datos_json = json.loads(RUTA_OFERTAS.read_text(encoding='utf-8'))
else:
    datos_json = {}

datos_json['radar_mercadona'] = {
    'ultima_recarga': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'descartadas_automaticamente': ['Especialista de Mantenimiento de Obra Civil (333617)', 'Personal de supermercado / reparto / IT puro / legal'],
    'total_nuevas': len(nuevas_mercadona),
    'total_inscritas': len(inscritas_mercadona),
    'nuevas_ofertas': nuevas_mercadona,
    'candidaturas_inscritas': inscritas_mercadona,
    'enlaces_directos': enlaces_mercadona
}
RUTA_OFERTAS.write_text(json.dumps(datos_json, indent=2, ensure_ascii=False), encoding='utf-8')

print('=' * 85)
print(f'🛒 RADAR MERCADONA — VALENCIA (Actualizado: {datos_json["radar_mercadona"]["ultima_recarga"]})')
print('=' * 85)

print(f'\n🆕 NUEVAS OFERTAS EN MERCADONA VALENCIA ({len(nuevas_mercadona)}):')
if nuevas_mercadona:
    for i, of in enumerate(nuevas_mercadona, 1):
        print(f"  {i}. {of['titulo']} ({of['municipio']})")
        print(f"     💰 {of['salario']} | Fuentes: {', '.join(of['fuentes']) or 'Portal Mercadona'}")
        if of['url_avature']:
            print(f"     🔗 Portal Mercadona: {of['url_avature']}")
        if of['url_linkedin']:
            print(f"     🔗 LinkedIn: {of['url_linkedin']}")
else:
    print('  • No hay nuevas vacantes técnicas/oficina adicionales a las 3 en las que ya estás inscrito hoy.')

print(f'\n✅ MIS CANDIDATURAS YA INSCRITAS EN MERCADONA ({len(inscritas_mercadona)}):')
for i, of in enumerate(inscritas_mercadona, 1):
    fuentes_txt = ', '.join(of['fuentes']) if of['fuentes'] else 'Seguimiento guardado'
    print(f"  {i}. [{of['estado']}] {of['titulo']}")
    print(f"     📍 {of['municipio']} | 💰 {of['salario']} | Activa en: {fuentes_txt}")
    if of['url_avature']:
        print(f"     🔗 Portal Mercadona: {of['url_avature']}")
    if of['url_linkedin']:
        print(f"     🔗 LinkedIn: {of['url_linkedin']}")

print('\n🔗 Enlaces directos al Radar Mercadona:')
for nombre, link in enlaces_mercadona.items():
    print(f'   • {nombre}:\n     {link}')


🛒 RADAR MERCADONA — VALENCIA (Actualizado: 2026-11-24 16:32:14)

🆕 NUEVAS OFERTAS EN MERCADONA VALENCIA (0):
  • No hay nuevas vacantes técnicas/oficina adicionales a las 3 en las que ya estás inscrito hoy.

✅ MIS CANDIDATURAS YA INSCRITAS EN MERCADONA (3):
  1. [✅ YA INSCRITO (2026-11-24)] Especialista en Instalaciones contra incendios y tratamiento del agua
     📍 Paterna | 💰 29. 678€ hasta 45.054€ brutos/año | Activa en: Avature
     🔗 Portal Mercadona: https://mercadona.avature.net/es_ES/Careers/JobDetail/Especialista-en-Instalaciones-contra-incendios-y-tratamiento-del-agua/333620
     🔗 LinkedIn: https://es.linkedin.com/jobs/view/especialista-en-instalaciones-contra-incendios-y-tratamiento-del-agua-at-mercadona-4481877744
  2. [✅ YA INSCRITO (2026-11-24)] Técnico/a de Automatización Industrial
     📍 Albalat dels Sorells / Valencia | 💰 29.678€ a 45.154€ | Activa en: Avature, LinkedIn
     🔗 Portal Mercadona: https://mercadona.avature.net/es_ES/Careers/JobDetail/T-cnico-a-de-Automa

## Paso 7: 🛍️ Radar Inditex — Parc Sagunt (Tempe / Bershka / Inditex)
Esta celda monitoriza el nuevo macro-hub logístico de **Inditex en Parc Sagunt** (>660.000 m² entre **Parc Sagunt I** —plataformas de **Tempe** ~141.000 m² y **Bershka** ~116.000 m², con finalización de obras prevista hacia mediados de 2027— y **Parc Sagunt II** —>381.000 m² junto a PowerCo—).

- Busca en directo en **LinkedIn (Valencia / Sagunto)** vacantes técnicas, de automatización, mantenimiento, electricidad/mecánica, ingeniería y logística/oficina en empresas del grupo (**Inditex, Tempe, Bershka, Zara, Pull&Bear, Stradivarius, Massimo Dutti, Oysho, Lefties**).
- **Descarta automáticamente** puestos de tienda y venta al público (*dependiente/a, vendedor/a, tienda, cajero/a, visual*).
- Pulsa ▶️ **Run** en cualquier momento para comprobar si ya han abierto procesos técnicos para Sagunto.

In [1]:
import json
import re
import urllib.parse
from datetime import datetime
from pathlib import Path
import requests
from bs4 import BeautifulSoup

RUTA_OFERTAS = Path('../ofertas/ofertas_guardadas.json')

EMPRESAS_INDITEX = [
    'inditex', 'tempe', 'bershka', 'pull&bear', 'zara',
    'massimo dutti', 'stradivarius', 'oysho', 'lefties'
]

EXCLUIR_TIENDA_INDITEX = [
    'dependient', 'vendedor', 'tienda', 'cajer', 'estilist', 'visual',
    'sales assistant', 'store', 'shop assistant', 'escaparat',
    'atención al cliente', 'responsable de sección'
]

KEYWORDS_TECNICO_LOGISTICA = [
    'automatiz', 'mantenim', 'eléctric', 'electric', 'mecánic', 'mecanic',
    'electromec', 'ingenier', 'oficina', 'técnic', 'tecnic', 'logístic',
    'logistic', 'instalacion', 'proyect', 'plc', 'robótic', 'robotic',
    'cad', 'eplan', 'operaciones', 'producción', 'produccion', 'industrial', 'sagunt'
]

HEADERS_RADAR = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept-Language': 'es-ES,es;q=0.9'
}

def buscar_radar_inditex():
    consultas = [
        'Inditex Sagunto',
        'Tempe Sagunto',
        'Bershka Sagunto',
        'Inditex automatizacion OR mantenimiento OR ingenieria OR logistica OR oficina',
        'Tempe automatizacion OR mantenimiento OR ingenieria OR logistica OR oficina'
    ]
    ofertas_unicas = {}
    ubicacion = urllib.parse.quote('Valencia, Comunidad Valenciana, España')

    for kw in consultas:
        url_li = (
            'https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search'
            f'?keywords={urllib.parse.quote(kw)}&location={ubicacion}&start=0'
        )
        try:
            r = requests.get(url_li, headers=HEADERS_RADAR, timeout=12)
            if r.status_code != 200:
                continue
            soup = BeautifulSoup(r.text, 'html.parser')
            for card in soup.find_all('div', class_='base-card'):
                t_el = card.find('h3', class_='base-search-card__title')
                c_el = card.find('h4', class_='base-search-card__subtitle')
                l_el = card.find('span', class_='job-search-card__location')
                a_el = card.find('a', class_='base-card__full-link')

                empresa = c_el.get_text(strip=True) if c_el else ''
                titulo = t_el.get_text(strip=True) if t_el else ''
                lugar = l_el.get_text(strip=True) if l_el else 'Valencia / Sagunto'
                enlace = a_el.get('href', '').split('?')[0] if a_el else ''

                if not enlace or not any(emp in empresa.lower() for emp in EMPRESAS_INDITEX):
                    continue

                low = f'{titulo} {empresa} {lugar}'.lower()
                if any(ex in low for ex in EXCLUIR_TIENDA_INDITEX):
                    continue
                if not any(k in low for k in KEYWORDS_TECNICO_LOGISTICA):
                    continue

                if enlace not in ofertas_unicas:
                    ofertas_unicas[enlace] = {
                        'titulo': titulo,
                        'empresa': empresa,
                        'ubicacion': lugar,
                        'url_linkedin': enlace,
                        'consulta_origen': kw
                    }
        except Exception as e:
            print(f'ℹ️ Nota consulta Inditex ({kw}): {e}')

    return list(ofertas_unicas.values())

ofertas_inditex = buscar_radar_inditex()

enlaces_inditex = {
    'Inditex Careers (Portal Oficial)': 'https://www.inditexcareers.com/portalweb/es/ofertas',
    'Inditex People': 'https://www.inditexpeople.com/',
    'Tempe (Grupo Inditex)': 'https://www.tempe.es/',
    'Sagunto Emplea (Portal Municipal)': 'https://sagunto.portalemp.com/',
    'LinkedIn Inditex / Tempe / Bershka (Sagunto)': 'https://www.linkedin.com/jobs/search/?keywords=Inditex%20OR%20Tempe%20OR%20Bershka&location=Sagunto%2C%20Comunidad%20Valenciana%2C%20Espa%C3%B1a'
}

if RUTA_OFERTAS.exists():
    datos_json = json.loads(RUTA_OFERTAS.read_text(encoding='utf-8'))
else:
    datos_json = {}

datos_json['radar_inditex_sagunto'] = {
    'ultima_recarga': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'estado_proyecto': 'Macro-hub logístico Parc Sagunt I (Tempe ~141.000 m² y Bershka ~116.000 m², fin de obras previsto mediados de 2027) y Parc Sagunt II (>381.000 m² junto a PowerCo).',
    'total_ofertas_tecnicas': len(ofertas_inditex),
    'ofertas_encontradas': ofertas_inditex,
    'enlaces_directos': enlaces_inditex
}
RUTA_OFERTAS.write_text(json.dumps(datos_json, indent=2, ensure_ascii=False), encoding='utf-8')

print('=' * 85)
print(f"🛍️ RADAR INDITEX — PARC SAGUNT (Actualizado: {datos_json['radar_inditex_sagunto']['ultima_recarga']})")
print('=' * 85)
print('ℹ️ Estado del bloque logístico:')
print('   Parc Sagunt I (Tempe y Bershka, obras hasta ~2027) + Parc Sagunt II (>381.000 m²).')

print(f'\n🔎 VACANTES TÉCNICAS / LOGÍSTICA / OFICINA ENCONTRADAS ({len(ofertas_inditex)}):')
if ofertas_inditex:
    for i, of in enumerate(ofertas_inditex, 1):
        print(f"  {i}. {of['titulo']} — {of['empresa']} ({of['ubicacion']})")
        print(f"     🔗 LinkedIn: {of['url_linkedin']}")
else:
    print('  • De momento hay 0 vacantes técnicas/logísticas publicadas en Sagunto/Valencia mientras avanzan las obras.')
    print('  • Vuelve a pulsar ▶️ Run periódicamente para detectar en cuanto abran mantenimiento, automatización o logística.')

print('\n🔗 Enlaces directos para vigilar Inditex / Tempe / Sagunto:')
for nombre, link in enlaces_inditex.items():
    print(f'   • {nombre}:\n     {link}')


🛍️ RADAR INDITEX — PARC SAGUNT (Actualizado: 2026-11-24 16:52:06)
ℹ️ Estado del bloque logístico:
   Parc Sagunt I (Tempe y Bershka, obras hasta ~2027) + Parc Sagunt II (>381.000 m²).

🔎 VACANTES TÉCNICAS / LOGÍSTICA / OFICINA ENCONTRADAS (0):
  • De momento hay 0 vacantes técnicas/logísticas publicadas en Sagunto/Valencia mientras avanzan las obras.
  • Vuelve a pulsar ▶️ Run periódicamente para detectar en cuanto abran mantenimiento, automatización o logística.

🔗 Enlaces directos para vigilar Inditex / Tempe / Sagunto:
   • Inditex Careers (Portal Oficial):
     https://www.inditexcareers.com/portalweb/es/ofertas
   • Inditex People:
     https://www.inditexpeople.com/
   • Tempe (Grupo Inditex):
     https://www.tempe.es/
   • Sagunto Emplea (Portal Municipal):
     https://sagunto.portalemp.com/
   • LinkedIn Inditex / Tempe / Bershka (Sagunto):
     https://www.linkedin.com/jobs/search/?keywords=Inditex%20OR%20Tempe%20OR%20Bershka&location=Sagunto%2C%20Comunidad%20Valenciana%2C%2

## Paso 8: 🔋 Radar Volkswagen / PowerCo — Gigafactoría Sagunto y Valencia
Esta celda consulta en directo el **portal oficial de carreras de PowerCo (`careers.powerco.de`)** en **Sagunto y Valencia** y lo cruza con **LinkedIn** para unificar todas las vacantes activas del proyecto de gigafactoría del Grupo Volkswagen.

- Excluye automáticamente puestos ajenos a tu perfil (*Security Officer*, RRHH, legal, finanzas, marketing).
- Clasifica las ofertas según tu CV (**Carlos Molina Gil — Doble FP Superior + Dositec / Ford / MetalWorld**):
  1. **`🎯 ENCAJE DIRECTO FP SUPERIOR`**: Mantenimiento y Producción industrial (electricidad, automatización, electromecánica, robótica/PLC).
  2. **`⚡ PERFIL OFICINA TÉCNICA / AUTOMATIZACIÓN / INGENIERÍA`**: Automatización, puesta en marcha y diseño de instalaciones (funcionalmente muy alineados con Dositec/EPLAN/AutoCAD, aunque PowerCo suele pedir Grado en Ingeniería e inglés alto en nivel *Expert*).
  3. **`📋 BOLSA DE TALENTO Y OTRAS VACANTES TÉCNICAS`**: Talent Pool oficial PowerCo Spain y otros puestos industriales/logísticos.

In [2]:
import json
import re
import urllib.parse
from datetime import datetime
from pathlib import Path
import requests
from bs4 import BeautifulSoup

RUTA_OFERTAS = Path('../ofertas/ofertas_guardadas.json')

HEADERS_POWERCO = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8'
}

EXCLUIR_POWERCO = [
    'security officer', 'human resources', 'hr ', 'recruiter',
    'legal', 'counsel', 'finance', 'accounting', 'communications',
    'marketing', 'payroll'
]

EQUIVALENCIAS_TITULOS = {
    'operario/a de mantenimiento': 'maintenance shopfloor operator',
    'operario/a de produccion': 'production shopfloor operator',
    'operario/a de logistica': 'logistics & warehouse shopfloor operator',
    'operario/a de calidad': 'quality & laboratory shopfloor operator',
}

def normalizar_texto_pc(texto):
    t = texto.lower()
    for orig, rep in [('á', 'a'), ('é', 'e'), ('í', 'i'), ('ó', 'o'), ('ú', 'u'), ('ñ', 'n')]:
        t = t.replace(orig, rep)
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clave_emparejamiento(titulo):
    low = titulo.lower().replace('ó', 'o').replace('í', 'i').replace('á', 'a').replace('é', 'e').replace('ú', 'u')
    for es_key, en_key in EQUIVALENCIAS_TITULOS.items():
        if es_key in low:
            return normalizar_texto_pc(en_key)
    return normalizar_texto_pc(titulo)

def clasificar_powerco(titulo):
    low = titulo.lower()
    es_directo = any(k in low for k in [
        'maintenance shopfloor', 'mantenimiento', 'production shopfloor',
        'producción', 'produccion', 'electromec', 'technician', 'técnico', 'tecnico'
    ]) and not any(k in low for k in ['expert', 'manager', 'director', 'lead'])
    if es_directo:
        return 'encaje_directo_fp'

    if any(k in low for k in ['talent pool', 'bolsa de talento']):
        return 'bolsa_talento'

    es_oficina_ing = any(k in low for k in [
        'commissioning', 'automation', 'facility design', 'infrastructure',
        'engineering', 'engineer', 'electrical', 'mechanical', 'plc',
        'robot', 'eplan', 'cad', 'process optimization'
    ])
    if es_oficina_ing:
        return 'oficina_automatizacion_ingenieria'

    return 'otras_vacantes_tecnicas'

def buscar_radar_powerco():
    ofertas = {}

    # 1. Portal oficial PowerCo (Sagunto y Valencia)
    for loc_query in ['Sagunto', 'Valencia']:
        url_portal = f'https://careers.powerco.de/search/?q=&locationsearch={urllib.parse.quote(loc_query)}'
        try:
            r = requests.get(url_portal, headers=HEADERS_POWERCO, timeout=15)
            if r.status_code != 200:
                print(f'ℹ️ Nota portal PowerCo ({loc_query}): HTTP {r.status_code}')
                continue
            soup = BeautifulSoup(r.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                href = a.get('href', '')
                if '/Power_Holdco_Spain/job/' not in href:
                    continue
                href_clean = href.split('?')[0]
                full_url = (
                    href_clean if href_clean.startswith('http')
                    else f'https://careers.powerco.de{href_clean}'
                )
                m_id = re.search(r'/(\d+)/?$', href_clean)
                job_id = m_id.group(1) if m_id else href_clean
                titulo = re.sub(r'\s+', ' ', a.get_text(' ', strip=True)).strip()
                if not titulo:
                    continue
                if any(ex in titulo.lower() for ex in EXCLUIR_POWERCO):
                    continue

                ubicacion = 'Sagunto' if '/Sagunto-' in href_clean or loc_query == 'Sagunto' else 'Valencia'
                if job_id not in ofertas:
                    ofertas[job_id] = {
                        'id': job_id,
                        'titulo': titulo,
                        'empresa': 'PowerCo (Grupo Volkswagen)',
                        'ubicacion': ubicacion,
                        'categoria_cv': clasificar_powerco(titulo),
                        'url_oficial': full_url,
                        'url_linkedin': '',
                        'fuentes': ['Portal PowerCo']
                    }
        except Exception as e:
            print(f'ℹ️ Nota portal PowerCo ({loc_query}): {e}')

    # 2. LinkedIn PowerCo en Valencia/Sagunto
    cards_li = []
    ubic_li = urllib.parse.quote('Valencia, Comunidad Valenciana, España')
    for start in [0, 25]:
        url_li = (
            'https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search'
            f'?keywords=PowerCo&location={ubic_li}&start={start}'
        )
        try:
            r_li = requests.get(url_li, headers=HEADERS_POWERCO, timeout=15)
            if r_li.status_code != 200:
                continue
            soup_li = BeautifulSoup(r_li.text, 'html.parser')
            for card in soup_li.find_all('div', class_='base-card'):
                t_el = card.find('h3', class_='base-search-card__title')
                c_el = card.find('h4', class_='base-search-card__subtitle')
                l_el = card.find('span', class_='job-search-card__location')
                a_el = card.find('a', class_='base-card__full-link')

                empresa = c_el.get_text(strip=True) if c_el else ''
                if 'powerco' not in empresa.lower():
                    continue
                titulo = t_el.get_text(strip=True) if t_el else ''
                if not titulo or any(ex in titulo.lower() for ex in EXCLUIR_POWERCO):
                    continue
                lugar = l_el.get_text(strip=True) if l_el else 'Valencia / Sagunto'
                enlace = a_el.get('href', '').split('?')[0] if a_el else ''
                m_li = re.search(r'-(\d+)/?$', enlace)
                li_num = int(m_li.group(1)) if m_li else 0
                cards_li.append({
                    'li_num': li_num,
                    'li_id': m_li.group(1) if m_li else enlace,
                    'titulo': titulo,
                    'lugar': lugar,
                    'enlace': enlace,
                    'clave': clave_emparejamiento(titulo)
                })
        except Exception as e:
            print(f'ℹ️ Nota LinkedIn PowerCo (start={start}): {e}')

    # Ordenar de más reciente (ID mayor) a más antiguo para priorizar enlaces vigentes en español/última publicación
    cards_li.sort(key=lambda x: x['li_num'], reverse=True)

    for item in cards_li:
        matched_key = None
        for k_id, off in ofertas.items():
            off_clave = clave_emparejamiento(off['titulo'])
            if item['clave'] == off_clave or item['clave'] in off_clave or off_clave in item['clave']:
                matched_key = k_id
                break

        if matched_key:
            off = ofertas[matched_key]
            if 'LinkedIn' not in off['fuentes']:
                off['fuentes'].append('LinkedIn')
            if not off['url_linkedin']:
                off['url_linkedin'] = item['enlace']
            # Si LinkedIn trae título en español para un puesto de mantenimiento/producción, mostrar ambos
            if any(x in item['titulo'].lower() for x in ['operario', 'técnico', 'tecnico']) and '/' not in off['titulo']:
                off['titulo'] = f"{off['titulo']} ({item['titulo']})"
        else:
            li_key = f"li_{item['li_id']}"
            ofertas[li_key] = {
                'id': item['li_id'],
                'titulo': item['titulo'],
                'empresa': 'PowerCo (Grupo Volkswagen)',
                'ubicacion': item['lugar'],
                'categoria_cv': clasificar_powerco(item['titulo']),
                'url_oficial': '',
                'url_linkedin': item['enlace'],
                'fuentes': ['LinkedIn']
            }

    lista = list(ofertas.values())

    def prioridad_directo(of):
        t = of['titulo'].lower()
        if 'maintenance' in t or 'mantenimiento' in t:
            return (0, t)
        if 'production' in t or 'producción' in t or 'produccion' in t:
            return (1, t)
        return (2, t)

    def prioridad_oficina(of):
        t = of['titulo'].lower()
        if 'commissioning' in t or 'automation' in t:
            return (0, t)
        if 'facility design' in t or 'infrastructure' in t:
            return (1, t)
        return (2, t)

    directas = sorted([o for o in lista if o['categoria_cv'] == 'encaje_directo_fp'], key=prioridad_directo)
    oficina = sorted([o for o in lista if o['categoria_cv'] == 'oficina_automatizacion_ingenieria'], key=prioridad_oficina)
    bolsa = [o for o in lista if o['categoria_cv'] == 'bolsa_talento']
    otras = sorted([o for o in lista if o['categoria_cv'] == 'otras_vacantes_tecnicas'], key=lambda o: o['titulo'].lower())
    return directas, oficina, bolsa, otras

directas_pc, oficina_pc, bolsa_pc, otras_pc = buscar_radar_powerco()

enlaces_powerco = {
    'Portal Oficial PowerCo — Sagunto': 'https://careers.powerco.de/search/?q=&locationsearch=Sagunto',
    'Portal Oficial PowerCo — Valencia': 'https://careers.powerco.de/search/?q=&locationsearch=Valencia',
    'LinkedIn PowerCo (Sagunto / Valencia)': 'https://www.linkedin.com/jobs/search/?keywords=PowerCo&location=Valencia%2C%20Comunidad%20Valenciana%2C%20Espa%C3%B1a'
}

if RUTA_OFERTAS.exists():
    datos_json = json.loads(RUTA_OFERTAS.read_text(encoding='utf-8'))
else:
    datos_json = {}

datos_json['radar_volkswagen_powerco'] = {
    'ultima_recarga': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'totales': {
        'encaje_directo_fp': len(directas_pc),
        'oficina_automatizacion_ingenieria': len(oficina_pc),
        'bolsa_talento': len(bolsa_pc),
        'otras_vacantes_tecnicas': len(otras_pc),
        'total_activas_filtradas': len(directas_pc) + len(oficina_pc) + len(bolsa_pc) + len(otras_pc)
    },
    'encaje_directo_fp': directas_pc,
    'oficina_automatizacion_ingenieria': oficina_pc,
    'bolsa_talento': bolsa_pc,
    'otras_vacantes_tecnicas': otras_pc,
    'enlaces_directos': enlaces_powerco
}
RUTA_OFERTAS.write_text(json.dumps(datos_json, indent=2, ensure_ascii=False), encoding='utf-8')

def imprimir_bloque_powerco(titulo_bloque, lista_of):
    print(f'\n{titulo_bloque} ({len(lista_of)}):')
    if not lista_of:
        print('  • Ninguna oferta activa en esta categoría en este momento.')
        return
    for i, of in enumerate(lista_of, 1):
        fuentes = ', '.join(of['fuentes'])
        print(f"  {i}. {of['titulo']} — {of['ubicacion']} | Fuentes: {fuentes}")
        if of['url_oficial']:
            print(f"     🔗 Portal PowerCo: {of['url_oficial']}")
        if of['url_linkedin']:
            print(f"     🔗 LinkedIn: {of['url_linkedin']}")

print('=' * 85)
print(f"🔋 RADAR VOLKSWAGEN / POWERCO — SAGUNTO Y VALENCIA (Actualizado: {datos_json['radar_volkswagen_powerco']['ultima_recarga']})")
print('=' * 85)

imprimir_bloque_powerco('🎯 1. ENCAJE DIRECTO CON TU CV (FP Superior + Experiencia Industrial)', directas_pc)
imprimir_bloque_powerco('⚡ 2. OFICINA TÉCNICA / AUTOMATIZACIÓN / INGENIERÍA (Perfil Expert — suele pedir Ingeniería e Inglés)', oficina_pc)
imprimir_bloque_powerco('📋 3. BOLSA DE TALENTO OFICIAL POWERCO SPAIN', bolsa_pc)
imprimir_bloque_powerco('🏭 4. OTRAS VACANTES TÉCNICAS / LOGÍSTICAS ACTIVAS EN SAGUNTO Y VALENCIA', otras_pc)

print('\n🔗 Enlaces directos al Radar PowerCo / Volkswagen:')
for nombre, link in enlaces_powerco.items():
    print(f'   • {nombre}:\n     {link}')


🔋 RADAR VOLKSWAGEN / POWERCO — SAGUNTO Y VALENCIA (Actualizado: 2026-11-24 16:52:14)

🎯 1. ENCAJE DIRECTO CON TU CV (FP Superior + Experiencia Industrial) (2):
  1. Maintenance Shopfloor Operator — Sagunto | Fuentes: Portal PowerCo, LinkedIn
     🔗 Portal PowerCo: https://careers.powerco.de/Power_Holdco_Spain/job/Sagunto-Maintenance-Shopfloor-Operator/1272296901/
     🔗 LinkedIn: https://es.linkedin.com/jobs/view/maintenance-shopfloor-operator-at-powerco-4439314147
  2. Production Shopfloor Operator — Sagunto | Fuentes: Portal PowerCo, LinkedIn
     🔗 Portal PowerCo: https://careers.powerco.de/Power_Holdco_Spain/job/Sagunto-Production-Shopfloor-Operator/1272298701/
     🔗 LinkedIn: https://es.linkedin.com/jobs/view/production-shopfloor-operator-at-powerco-4340384331

⚡ 2. OFICINA TÉCNICA / AUTOMATIZACIÓN / INGENIERÍA (Perfil Expert — suele pedir Ingeniería e Inglés) (4):
  1. Automation Process Optimization Expert — Valencia | Fuentes: Portal PowerCo
     🔗 Portal PowerCo: https://care